# 🎯 Arabic Pronunciation Assessment – Production Backend

**State-of-the-art Pipeline** (fest konfiguriert, keine Alternativen):

| Stufe | Tool | Zweck |
|---|---|---|
| VAD / Trim | **Silero VAD 5** | Stille am Anfang/Ende entfernen |
| ASR | **wav2vec 2.0 XLSR-53 Arabic** | Beste offene Arabisch-Erkennung |
| Alignment | **`torchaudio.functional.forced_align`** | CUDA-beschleunigte CTC-Alignment |
| Scoring | **GOP** (mittlere Log-Prob pro Buchstabe) | 0 – 100 pro Buchstabe |
| Serving | **FastAPI + Uvicorn**, Pydantic-Validierung | Getypter Endpoint mit Limits |
| Tunnel | **Cloudflared Quick Tunnel** | Öffentliche URL ohne Login |

**Ausführung in Colab:**
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`
3. Letzte Zelle druckt die URL für `index.html`.

In [ ]:
%pip install -q -U transformers pydub nest_asyncio python-multipart
%pip install -q -U fastapi 'uvicorn[standard]'
%pip install -q -U silero-vad

!apt-get -qq install -y ffmpeg > /dev/null
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

import importlib
for mod in ("transformers", "pydub", "fastapi", "uvicorn", "silero_vad", "torch", "torchaudio"):
    m = importlib.import_module(mod)
    print(f"  ✅ {mod:14s} {getattr(m, '__version__', '?')}")
print("✅ Alle Abhängigkeiten importierbar.")

In [ ]:
import io, os, re, time, subprocess, threading, urllib.request, unicodedata
from typing import List, Dict, Any

import numpy as np
import torch
import torchaudio.functional as AF
from pydub import AudioSegment
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from silero_vad import load_silero_vad, get_speech_timestamps

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SR = 16000
print(f"✅ Device: {device}, torch {torch.__version__}")

In [ ]:
ASR_MODEL_ID = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"

print("Lade ASR-Modell …")
asr_processor = Wav2Vec2Processor.from_pretrained(ASR_MODEL_ID)
asr_model     = Wav2Vec2ForCTC.from_pretrained(ASR_MODEL_ID).to(device).eval()
ASR_VOCAB    = asr_processor.tokenizer.get_vocab()
ASR_BLANK_ID = asr_model.config.pad_token_id
print(f"  ✅ wav2vec2 XLSR-53 Arabic  ({len(ASR_VOCAB)} tokens, blank={ASR_BLANK_ID})")

print("Lade Silero VAD …")
vad_model = load_silero_vad()
print("  ✅ Silero VAD 5")

with torch.inference_mode():
    _ = asr_model(torch.zeros(1, SR, device=device)).logits
print("✅ Modelle geladen und aufgewärmt.")

In [ ]:
def decode_audio(raw: bytes) -> np.ndarray:
    """Beliebiges Audioformat -> 16 kHz mono float32, unverändert."""
    seg = AudioSegment.from_file(io.BytesIO(raw))
    seg = seg.set_frame_rate(SR).set_channels(1).set_sample_width(2)
    return np.asarray(seg.get_array_of_samples(), dtype=np.float32) / 32768.0

def gentle_trim(audio: np.ndarray, pad_ms: int = 250) -> np.ndarray:
    """Nur führende/nachlaufende lange Stille entfernen. Rest bleibt unangetastet."""
    segs = get_speech_timestamps(torch.from_numpy(audio), vad_model,
                                 sampling_rate=SR, threshold=0.35)
    if not segs:
        return audio
    pad = int(pad_ms * SR / 1000)
    start = max(0, segs[0]["start"] - pad)
    end   = min(len(audio), segs[-1]["end"] + pad)
    return audio[start:end]

def preprocess(raw: bytes) -> np.ndarray:
    return gentle_trim(decode_audio(raw))

In [ ]:
# Nur klassisches Tashkeel entfernen. Hamza-Formen (أ إ آ ؤ ئ) bleiben als eigene Buchstaben erhalten.
_TASHKEEL = set("ًٌٍَُِّْٰ")

def strip_diacritics(text: str) -> str:
    nfd = unicodedata.normalize("NFD", text)
    return unicodedata.normalize("NFC", "".join(c for c in nfd if c not in _TASHKEEL))

def encode_target(word: str) -> List[int]:
    ids: List[int] = []
    for ch in word:
        tid = ASR_VOCAB.get(ch)
        if tid is None:
            raise ValueError(f"Zeichen {ch!r} nicht im ASR-Vokabular.")
        ids.append(tid)
    return ids

@torch.inference_mode()
def run_asr(audio: np.ndarray):
    inputs = asr_processor(audio, sampling_rate=SR, return_tensors="pt", padding=True)
    logits = asr_model(inputs.input_values.to(device)).logits
    log_probs = torch.log_softmax(logits, dim=-1).cpu()
    transcription = asr_processor.batch_decode(log_probs.argmax(dim=-1))[0]
    return log_probs, transcription

def _runs_of_non_blank(tokens: List[int]) -> List[List[int]]:
    runs: List[List[int]] = []
    current: List[int] = []
    last: int = -1
    for t, tok in enumerate(tokens):
        if tok == ASR_BLANK_ID:
            if current: runs.append(current); current = []
            last = -1
        elif tok != last:
            if current: runs.append(current)
            current = [t]; last = tok
        else:
            current.append(t)
    if current: runs.append(current)
    return runs

def gop_score(log_probs: torch.Tensor, target_word: str) -> List[Dict[str, Any]]:
    target_ids = encode_target(target_word)
    if not target_ids:
        return []
    if log_probs.shape[1] < len(target_ids):
        raise ValueError("Aufnahme zu kurz für dieses Wort.")
    targets = torch.tensor([target_ids], dtype=torch.int32)
    aligned, _ = AF.forced_align(log_probs, targets, blank=ASR_BLANK_ID)
    runs = _runs_of_non_blank(aligned[0].tolist())
    results: List[Dict[str, Any]] = []
    for i, (ch, tid) in enumerate(zip(target_word, target_ids)):
        if i < len(runs):
            frames = runs[i]
            mean_lp = log_probs[0, frames, tid].mean().item()
            score = float(np.clip((mean_lp + 3.0) / 3.0 * 100, 0, 100))
            conf  = float(np.exp(mean_lp))
        else:
            score, conf = 0.0, 0.0
        results.append({"label": ch, "score": score, "confidence": conf})
    return results

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

MAX_AUDIO_BYTES = 3 * 1024 * 1024
MIN_SAMPLES     = int(0.15 * SR)

class Unit(BaseModel):
    label: str
    score: float
    confidence: float

class AssessResponse(BaseModel):
    target: str
    transcription: str
    units: List[Unit]
    total: float
    duration_ms: int

app = FastAPI(title="Arabic Pronunciation API", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

@app.get("/health")
def health():
    return {"status": "ok", "device": str(device),
            "asr_model": ASR_MODEL_ID, "vad": "Silero VAD 5"}

@app.post("/assess", response_model=AssessResponse)
def assess(audio: UploadFile = File(...), target: str = Form(...)):
    target = target.strip()
    if not target:
        raise HTTPException(400, "Zielwort fehlt.")
    t0 = time.perf_counter()
    raw = audio.file.read(MAX_AUDIO_BYTES + 1)
    if len(raw) > MAX_AUDIO_BYTES:
        raise HTTPException(413, f"Audio > {MAX_AUDIO_BYTES // 1024} KB.")
    if not raw:
        raise HTTPException(400, "Leere Audiodatei.")
    try:
        wav = preprocess(raw)
    except Exception as e:
        raise HTTPException(400, f"Audio ungültig: {e}")
    if wav.size < MIN_SAMPLES:
        raise HTTPException(400, "Aufnahme zu kurz.")
    target_clean = strip_diacritics(target)
    try:
        log_probs, transcription = run_asr(wav)
        units = gop_score(log_probs, target_clean)
    except ValueError as e:
        raise HTTPException(400, str(e))
    total = float(np.mean([u["score"] for u in units])) if units else 0.0
    return AssessResponse(
        target=target_clean,
        transcription=transcription,
        units=[Unit(**u) for u in units],
        total=total,
        duration_ms=int((time.perf_counter() - t0) * 1000),
    )

print("✅ API definiert:  GET /health   POST /assess")

In [ ]:
import uvicorn, nest_asyncio
nest_asyncio.apply()

PORT   = 8000
CF_LOG = "/tmp/cf.log"

threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT,
                               log_level="warning", access_log=False),
    daemon=True,
).start()

for _ in range(30):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=1).read()
        print(f"✅ FastAPI läuft auf Port {PORT}")
        break
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError("FastAPI-Start fehlgeschlagen.")

if os.path.exists(CF_LOG):
    os.remove(CF_LOG)
subprocess.Popen(f"cloudflared tunnel --url http://localhost:{PORT} > {CF_LOG} 2>&1 &", shell=True)

public_url = None
for _ in range(60):
    time.sleep(1)
    if os.path.exists(CF_LOG):
        m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", open(CF_LOG).read())
        if m:
            public_url = m.group(0)
            break

if not public_url:
    raise RuntimeError("Keine Tunnel-URL. Log:\n" + open(CF_LOG).read()[-800:])

print("\n" + "=" * 68)
print(f"🌍 Backend-URL für index.html:  {public_url}")
print("=" * 68)
print(f"Health-Check:  {public_url}/health")